# Feature Checks

First-order screen for leakage and low-quality features, plus a train→OOT persistence check, and where the exclude list is built. Reads the **candidate** projection so a feature stays visible even after exclusion. The screen **flags, it never drops**; there is **no automated selection**.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from configs._schema import load_config
cfg = load_config(ROOT / 'configs/fx_activation.yaml')   # validates on load

from src.dataset import load_labelled
from src.leakage import leakage_report
from src.feature_stats import signal_persistence

X, y = load_labelled(spark, cfg, cfg.obs_date, feature_cols=cfg.candidates)

## 1. FX Activation

### First-order screen (train month)
Per candidate: direction-agnostic univariate AUC, % missing, value dominance, and a `suspected_leak` flag (univariate AUC > 0.95).

In [ ]:
rep = leakage_report(X, y, cfg.candidates)
rep.head(20)

### Read the flags
High AUC → **investigate, don't reflexively drop**: real signal or a post-outcome proxy? High dominance / missingness → usually a drop.

In [ ]:
rep[rep['suspected_leak']]                                     # interrogate

In [ ]:
rep[(rep['value_dominance'] > 0.98) | (rep['pct_missing'] > 0.5)]   # usually drop

### Signal persistence (train → OOT)
The blind spot of any single-month screen is train→serve drift. This loads the **OOT** month and compares each feature's univariate AUC on train vs OOT, plus **PSI** (input drift). `flag = True` when signal decays (`auc_drop > 0.03`) or the feature drifts heavily (`psi > 0.25`). A feature that is strong in March but not June — or that drifts by August — is a liability to drop or monitor, regardless of how good it looks in-sample.

In [ ]:
X_oot, y_oot = load_labelled(spark, cfg, cfg.oot_date, feature_cols=cfg.candidates)
sp = signal_persistence(X, y, X_oot, y_oot, cfg.candidates)
sp.head(20)

### Human decision → exclude list
Combine the leakage flags and the persistence flags. Edit by hand, then paste the printed block into `features_exclude` in the config.

In [ ]:
candidate_excludes = [
    'fx_fwd_txn_3m',   # raw forward signal the target was built from — a leak
]
print('features_exclude:')
[print(f'  - {f}') for f in candidate_excludes]

### Redundancy (optional)
Correlation among survivors — complements the VIF view in EDA. Not a selection step.

In [ ]:
import numpy as np
surv = [f for f in cfg.candidates if f not in candidate_excludes]
corr = X[surv].select_dtypes('number').corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
         .stack().sort_values(ascending=False))
pairs[pairs > 0.9].head(15)